<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Attention_and_Prompted_probes_generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer_lens

# Setup files

Downloading necessary modules

In [ ]:
import transformer_lens
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm


import numpy as np
import pandas as pd
import os
import json
import requests
from pathlib import Path
from typing import List, Dict
from collections import Counter

import plotly.express as px
import matplotlib

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

## Downloading the Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B",
                                            torch_dtype="auto",
                                            )

## Downloading the Train and Test datasets

In [ ]:
DATA_DIR = Path("data/high_stakes")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/training/prompts_4x/train.jsonl"
train_path = DATA_DIR / "train.jsonl"

response = requests.get(train_url)
response.raise_for_status()

train_path.write_bytes(response.content)
print("Saved train data to", train_path)


anthropic_dev_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/evals/test/anthropic_test_balanced_apr_23.jsonl"
anthropic_dev_path = DATA_DIR / "anthropic_dev.jsonl"


response = requests.get(anthropic_dev_url)
response.raise_for_status()
anthropic_dev_path.write_bytes(response.content)

print('Saved test data to', anthropic_dev_path)


In [ ]:
def load_jsonl(path) -> List[Dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def label_to_int(x: str) -> int:
    if x == "high-stakes":
        return 1
    elif x == "low-stakes":
        return 0
    else:
        raise ValueError(f"Unexpected label: {x!r}")


def normalize_inputs(inputs_field: str) -> str:
    s = inputs_field.strip()


    if s.startswith('[') and '"role"' in s:
        try:
            messages = json.loads(s)
            parts = [f"{m['role']}: {m['content']}" for m in messages]
            return "\n".join(parts)
        except json.JSONDecodeError:

            return inputs_field
    else:

        return inputs_field


In [ ]:
train_rows = load_jsonl("data/high_stakes/train.jsonl")
dev_rows   = load_jsonl("data/high_stakes/anthropic_dev.jsonl")
len(train_rows), len(dev_rows)

In [ ]:
train_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in train_rows]
test_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in dev_rows]

train_texts = [train['text'] for train in train_dataset]
train_labels= [train['label'] for train in train_dataset]

test_texts = [test['text'] for test in test_dataset]
test_labels= [test['label'] for test in test_dataset]

In [ ]:
def create_dataloaders(
    activations: np.ndarray,
    labels: List[int],
    batch_size: int = 32,
    train_split: float = 0.8
):
    """Create train/val dataloaders from activations and labels"""

    # Convert to tensors
    X = torch.FloatTensor(activations)
    y = torch.FloatTensor(labels)

    # Create dataset
    dataset = TensorDataset(X, y)

    # Split train/val if needed
    if train_split < 1.0:
        train_size = int(train_split * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(
            dataset, [train_size, val_size]
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        return train_loader, val_loader
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        return loader


In [ ]:
model = transformer_lens.HookedTransformer.from_pretrained("Qwen/Qwen2.5-0.5B")

In [ ]:
def get_activations(texts, model, layer_idx=-1, batch_size=8, pooling='last', pad_all=True):
    """
    Extract activations with different pooling strategies.

    Args:
        pad_all: If True and pooling='all', pad sequences to same length
    """
    model.eval()
    all_activations = []

    hook_name = f'blocks.{layer_idx}.hook_resid_post'

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        for text in batch_texts:
            captured = []

            def hook_fn(activation, hook):
                captured.append(activation.clone().cpu())

            with torch.no_grad():
                model.run_with_hooks(
                    text,
                    fwd_hooks=[(hook_name, hook_fn)]
                )

            hidden_states = captured[0][0]  # [seq_len, d_model]

            if pooling == 'last':
                act = hidden_states[-1, :]
            elif pooling == 'mean':
                act = hidden_states.mean(dim=0)
            elif pooling == 'first':
                act = hidden_states[0, :]
            elif pooling == 'all':
                act = hidden_states  # [seq_len, d_model]

            all_activations.append(act)
            del captured, hidden_states, act

        torch.cuda.empty_cache()

    # Concatenate based on pooling
    if pooling == 'all':
        if pad_all:
            # Pad to same length
            from torch.nn.utils.rnn import pad_sequence
            padded = pad_sequence(all_activations, batch_first=True)
            return padded.numpy()  # [num_texts, max_seq_len, d_model]
        else:
            return all_activations  # List of varying length tensors
    else:
        return torch.stack(all_activations, dim=0).numpy()

In [ ]:
acts = get_activations(train_texts[:1000], model, layer_idx=1, batch_size=8, pooling='mean')